In [ ]:
# --- setup -------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

import os

REPO = '/content/pxr-repo'
if os.path.exists(REPO):
    !cd $REPO && git pull
else:
    !git clone https://github.com/pridem755/patient-or-xray.git $REPO
%pip install -q -e $REPO

In [ ]:
import importlib
import site
import sys

site.main()
importlib.invalidate_caches()
SRC = f'{REPO}/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pxr
print('pxr loaded from:', pxr.__file__)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from pxr.config import load_config
from pxr.data.splits import fold_membership
from pxr.stats.calibration import (
    apply_calibration,
    calibrate_fold,
    calibration_report,
    reliability_table,
)

cfg = load_config(f'{REPO}/config/study_config.yaml')
ROOT = Path(cfg.paths['drive_root'])
COHORTS = ROOT / cfg.paths['cohorts']
SPLITS = ROOT / cfg.paths['splits']
SCORES = ROOT / cfg.paths['scores']
CALIB = ROOT / cfg.paths['calibration']
CALIB.mkdir(parents=True, exist_ok=True)

rule = cfg.analysis['threshold_rule']
TARGET = float(rule['fixed_sensitivity_target'])
ARMS = cfg.analysis['calibration']['conditioning']

print('cohort / split / model hash:', cfg.cohort_hash, cfg.split_hash, cfg.model_hash)
print('threshold rule :', rule['primary'], f'at {TARGET:.0%} validation sensitivity')
print('sensitivity rule:', rule['sensitivity_rule'])
print('calibration arms:', ARMS)
print('labels :', len(cfg.analysis_labels))

In [ ]:
# --- load cohorts and out-of-fold predictions -----------------------------------
cohorts, oof = {}, {}
folds = pd.read_parquet(SPLITS / f'folds_{cfg.split_hash}.parquet')

for site in cfg.training_sites:
    cohort = pd.read_parquet(COHORTS / cfg.artifact_name('cohort', site=site))
    cohort = cohort.merge(folds[folds.site == site][['patient_id', 'fold']],
                          on='patient_id', how='left')
    cohorts[site] = cohort
    oof[site] = pd.read_parquet(SCORES / f'oof_{site}_{cfg.model_hash}.parquet')
    print(f'{site:<10} {len(cohort):>7,} patients, {len(oof[site]):>7,} out-of-fold scores')

In [ ]:
# --- load models and make validation predictions ---------------------------------
import torch

from pxr.data.cache import load_cache
from pxr.model.train import build_model, predict, training_config_from

train_cfg = training_config_from(cfg)
MODELS = ROOT / cfg.paths['models']
LOCAL_CACHE = Path('/content/cache')

def roles_for(site, fold):
    return fold_membership(
        fold, cohorts[site],
        stratify_by=cfg.stratify_by, val_stratify_by=cfg.val_stratify_by,
        n_folds=cfg.n_folds, val_fraction=cfg.val_fraction,
        min_val_stratum=cfg.min_val_stratum, seed=cfg.splits['seed'],
    )

caches = {}
for site in cfg.training_sites:
    local = LOCAL_CACHE / site
    if not local.exists():
        import shutil
        shutil.copytree(ROOT / cfg.paths['image_cache'] / site, local)
    caches[site] = load_cache(local, cohort_ids=cohorts[site]['image_id'],
                              config_hash=cfg.cohort_hash)

validation_scores = {}
for site in cfg.training_sites:
    path = SCORES / f'val_{site}_{cfg.model_hash}.parquet'
    if path.exists():
        validation_scores[site] = pd.read_parquet(path)
        print(f'{site}: validation scores loaded')
        continue

    per_fold = []
    for fold in range(cfg.n_folds):
        model = build_model(train_cfg.architecture, len(train_cfg.labels), pretrained=False)
        model.load_state_dict(torch.load(MODELS / f'{site}_fold{fold}.pt', map_location='cpu'))
        role = roles_for(site, fold)
        scored = predict(model, cohorts[site][role == 'val'], caches[site], train_cfg)
        per_fold.append(scored.assign(fold=fold))
        del model
        torch.cuda.empty_cache()
        print(f'  {site} fold {fold}: {len(scored):,} validation predictions')

    validation_scores[site] = pd.concat(per_fold, ignore_index=True)
    validation_scores[site].to_parquet(path, index=False)

In [ ]:
# --- assemble validation and out-of-fold predictions for analysis ---------------
def assemble(scores, cohort, labels):
    truth = cohort[['patient_id', 'view', 'sex', 'age', *labels]].rename(
        columns={l: f'{l}_true' for l in labels})
    merged = scores.rename(columns={l: f'{l}_score' for l in labels}).merge(
        truth, on='patient_id', how='inner')
    return merged

val_frames, test_frames = {}, {}
for site in cfg.training_sites:
    val_frames[site] = assemble(validation_scores[site], cohorts[site], cfg.analysis_labels)
    test_frames[site] = assemble(oof[site], cohorts[site], cfg.analysis_labels)
    print(f'{site:<10} validation {len(val_frames[site]):>7,}  '
          f'out-of-fold {len(test_frames[site]):>7,}')

In [ ]:
# --- calibrate and score folds ---------------------------------------------------
calibrations, decisions = {}, {}

for site in cfg.training_sites:
    for arm in ARMS:
        fitted, scored_folds = [], []
        for fold in range(cfg.n_folds):
            val_fold = val_frames[site][val_frames[site]['fold'] == fold]
            test_fold = test_frames[site][test_frames[site]['fold'] == fold]
            if val_fold.empty or test_fold.empty:
                raise RuntimeError(f'{site} fold {fold}: missing validation or test rows')

            calibration = calibrate_fold(
                val_fold, cfg.analysis_labels, fold=fold, arm=arm,
                target_sensitivity=TARGET, threshold_rule=rule['primary'],
            )
            fitted.append(calibration.to_frame())
            scored_folds.append(apply_calibration(test_fold, calibration,
                                                  cfg.analysis_labels))

        calibrations[(site, arm)] = pd.concat(fitted, ignore_index=True)
        decisions[(site, arm)] = pd.concat(scored_folds, ignore_index=True)
        print(f'{site:<10} {arm:<7} {len(decisions[(site, arm)]):>7,} patients scored')

In [ ]:
# --- summarize calibration and thresholding results -----------------------------
for site in cfg.training_sites:
    print(f'=== {site} ===')
    for arm in ARMS:
        frame = calibrations[(site, arm)]
        columns = [c for c in frame.columns if c.startswith('T_')]
        summary = frame.groupby('label')[columns].mean().round(3)
        summary['threshold'] = frame.groupby('label')['threshold'].mean().round(4)
        summary['achieved_sens'] = frame.groupby('label')['achieved_sensitivity'].mean().round(3)
        print(f'\n  arm = {arm} (mean across folds)')
        print(summary.reindex(cfg.analysis_labels).to_string())
    print()

In [ ]:
# does the AP/PA temperature gap look systematic, or like fold noise?
for site in cfg.training_sites:
    frame = calibrations[(site, 'view')]
    if not {'T_AP', 'T_PA'} <= set(frame.columns):
        continue
    gap = (frame['T_AP'] - frame['T_PA'])
    print(f'{site}: T_AP - T_PA across all fold-label pairs')
    print(f'  mean {gap.mean():+.3f}   sd {gap.std():.3f}   '
          f'range [{gap.min():+.3f}, {gap.max():+.3f}]')

In [ ]:
# --- generate calibration reports ------------------------------------------------
for site in cfg.training_sites:
    print(f'=== {site} ===')
    for arm in ARMS:
        report = calibration_report(decisions[(site, arm)], cfg.analysis_labels)
        print(f'\n  arm = {arm}')
        print(report.round(4).to_string(index=False))
    print()

In [ ]:
# --- generate reliability diagrams ------------------------------------------------
import matplotlib.pyplot as plt

site = cfg.training_sites[0]
label = cfg.analysis_labels[0]
frame = decisions[(site, 'global')]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, column, title in [(axes[0], f'{label}_score', 'raw'),
                          (axes[1], f'{label}_calibrated', 'calibrated')]:
    table = reliability_table(frame[column].to_numpy(float),
                             frame[f'{label}_true'].to_numpy(float))
    ax.plot([0, 1], [0, 1], ls='--', c='grey', lw=1)
    ax.plot(table['mean_predicted'], table['observed_rate'], marker='o')
    ax.set_xlabel('predicted probability'); ax.set_ylabel('observed rate')
    ax.set_title(f'{site} {label}: {title}')
plt.tight_layout(); plt.show()

In [ ]:
# --- save decisions and calibration results --------------------------------------
for (site, arm), frame in decisions.items():
    path = SCORES / f'decisions_{site}_{arm}_{cfg.model_hash}.parquet'
    frame.to_parquet(path, index=False)
    calibrations[(site, arm)].to_csv(
        CALIB / f'calibration_{site}_{arm}_{cfg.model_hash}.csv', index=False)
    print(f'{site:<10} {arm:<7} -> {path.name}')

In [ ]:
# --- integrity cell ------------------
print(f'model_hash : {cfg.model_hash}')
print(f'threshold rule : {rule["primary"]} at {TARGET:.0%} validation sensitivity')
print(f'arms : {ARMS}')
print('fitted per fold, per label; one threshold per fold-label pair applied to '
      'AP and PA alike\n')
for site in cfg.training_sites:
    for arm in ARMS:
        frame = decisions[(site, arm)]
        flagged = {l: float((frame[f'{l}_predicted'] == 1).mean())
                   for l in cfg.analysis_labels[:3]}
        print(f'{site:<10} {arm:<7} {len(frame):>7,} patients  '
              f'flagged: ' + ', '.join(f'{k} {v:.1%}' for k, v in flagged.items()))